# POC: S3R / SSM

📖 対応記事: `親子対話：S3R`🔗 [記事を読む](articles/親子対話：S3R.md)

In [ ]:
# @title Setup
!pip install -q torch numpy

In [ ]:
# @title Demo
# Structured State Space for Speech
import torch
import torch.nn as nn

# Minimal S4-inspired layer
class S4Layer(nn.Module):
  def __init__(self, dim=32):
    super().__init__()
    self.dim = dim
    # Hippo-like initialization (simplified)
    dt = 0.01
    A = -torch.eye(dim) * 0.5
    B = torch.ones(dim, 1)
    C = torch.ones(1, dim)
    self.register_buffer('A', A)
    self.register_buffer('B', B)
    self.register_buffer('C', C)
    self.linear = nn.Linear(dim, dim)
  
  def forward(self, x):
    B, L, D = x.shape
    h = torch.zeros(B, D, device=x.device)
    outputs = []
    for t in range(L):
      h = h @ self.A.T + x[:, t] @ self.B.T
      y = h @ self.C.T
      outputs.append(y)
    out = torch.stack(outputs, dim=1)
    return self.linear(out)

model = S4Layer(32)
x = torch.randn(2, 50, 32)
out = model(x)
print(f"S4 Layer: {x.shape} → {out.shape}")
print(f"✅ SSM processes 50-length sequence with constant memory")

---
undefined